# TDC-KV Colab Quickstart

Use this notebook for small Google Colab tests before running expensive 7B/8B experiments.

Recommended Colab runtime for this notebook:

- Runtime type: Python 3
- Hardware accelerator: GPU
- For trace tests: T4 is enough
- For tiny HF smoke tests: T4/L4 is enough
- For real 7B/8B tests: prefer L4/A100

Important: if you clone from GitHub, Colab only sees committed/pushed code. If your local working tree has changes that are not pushed, either push them first or use the zip-upload path below.

In [1]:
# 1. Check GPU and Python environment
import os, platform, subprocess, sys

print('Python:', sys.version)
print('Platform:', platform.platform())

try:
    import torch
    print('Torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('Torch check failed:', repr(exc))

subprocess.run(['nvidia-smi'], check=False)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
Torch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


CompletedProcess(args=['nvidia-smi'], returncode=0)

In [2]:
!nvidia-smi

Sat Aug  1 08:50:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Get The Code

Default path for Colab through the VS Code extension: clone `branch-h` from GitHub. Keep `USE_GITHUB = True` unless you need to test local changes that are not pushed yet. If you do need local unpushed changes, set `USE_GITHUB = False`, run the cell, and upload a `.zip` of the project folder from the browser Colab UI.

In [3]:
# 2. Get the project code
from pathlib import Path
import shutil

USE_GITHUB = True
REPO_URL = 'https://github.com/JayGor-13/Tier-based-KV-cache-with-Dependency-Aware-Chunk-Scoring.git'
REPO_BRANCH = 'branch-h'
PROJECT_DIR = Path('/content/Tier-based-KV-cache-with-Dependency-Aware-Chunk-Scoring')

if USE_GITHUB:
    shutil.rmtree(PROJECT_DIR, ignore_errors=True)
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
else:
    from google.colab import files
    import zipfile
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No zip uploaded.')
    zip_name = next(iter(uploaded.keys()))
    shutil.rmtree(PROJECT_DIR, ignore_errors=True)
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_name) as zf:
        zf.extractall(PROJECT_DIR)
    nested = [p for p in PROJECT_DIR.iterdir() if p.is_dir() and (p / 'src').exists()]
    if nested:
        PROJECT_DIR = nested[0]

os.chdir(PROJECT_DIR)
print('Project dir:', Path.cwd())
print('Files:', sorted(p.name for p in Path.cwd().iterdir())[:20])

Project dir: /content/Tier-based-KV-cache-with-Dependency-Aware-Chunk-Scoring
Files: ['.git', '.gitignore', 'README.md', 'benchmarks', 'context.md', 'data', 'environment.yml', 'notebooks', 'pyproject.toml', 'requirements.txt', 'scripts', 'specifications.md', 'src', 'tdc_kv_results_experiment_plan.md', 'tests']


In [4]:
# 3. Install lightweight dependencies
# Do not reinstall torch on Colab unless you have a specific CUDA reason.
packages = [
    'pytest',
    'numpy',
    'scipy',
    'matplotlib',
    'accelerate',
    'datasets',
    'transformers',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)

os.environ['PYTHONPATH'] = str(Path.cwd()) + os.pathsep + os.environ.get('PYTHONPATH', '')
print('PYTHONPATH starts with:', os.environ['PYTHONPATH'].split(os.pathsep)[0])

PYTHONPATH starts with: /content/Tier-based-KV-cache-with-Dependency-Aware-Chunk-Scoring


In [5]:
# 4. Import sanity check
import src.core
import src.baselines
from benchmarks.hf_runner import normalize_methods

print('imports ok')
print('methods:', normalize_methods(['fullkv', 'streamingllm', 'h2o', 'snapkv', 'chunkkv', 'tdc-kv']))

imports ok
methods: ['fullkv', 'streamingllm', 'h2o', 'snapkv', 'chunkkv', 'tdc_kv']


In [6]:
# 5. Run fast unit tests
subprocess.run([
    sys.executable, '-m', 'pytest',
    'tests/test_chunker.py',
    'tests/test_scorer.py',
    'tests/test_masker.py',
    'tests/test_evictor.py',
    'tests/test_baselines.py',
    '-q'
], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pytest', 'tests/test_chunker.py', 'tests/test_scorer.py', 'tests/test_masker.py', 'tests/test_evictor.py', 'tests/test_baselines.py', '-q'], returncode=0)

In [7]:
# 6. Run trace-driven smoke tests. These are cheap and should work even on T4.
Path('outputs').mkdir(exist_ok=True)

subprocess.run([
    sys.executable, 'scripts/run_main_results.py',
    '--trace-path', 'data/sample_trace.jsonl',
    '--recent-window', '4',
    '--output', 'outputs/colab_main_smoke.json'
], check=True)

subprocess.run([
    sys.executable, 'scripts/run_baselines.py',
    '--trace-path', 'data/sample_trace.jsonl',
    '--methods', 'streamingllm,chunkkv,snapkv,h2o',
    '--recent-window', '4',
    '--output', 'outputs/colab_baselines_smoke.json'
], check=True)

subprocess.run([
    sys.executable, 'scripts/run_ablations.py',
    '--trace-path', 'data/sample_trace.jsonl',
    '--theta-grid', '0.2,0.3',
    '--recent-window-grid', '4,8',
    '--output', 'outputs/colab_ablations_smoke.json'
], check=True)

CompletedProcess(args=['/usr/bin/python3', 'scripts/run_ablations.py', '--trace-path', 'data/sample_trace.jsonl', '--theta-grid', '0.2,0.3', '--recent-window-grid', '4,8', '--output', 'outputs/colab_ablations_smoke.json'], returncode=0)

In [8]:
# 7. Inspect smoke outputs
import json

for path in [
    'outputs/colab_main_smoke.json',
    'outputs/colab_baselines_smoke.json',
    'outputs/colab_ablations_smoke.json',
]:
    print('\n===', path, '===')
    with open(path, 'r', encoding='utf-8') as f:
        payload = json.load(f)
    if 'summary' in payload:
        print(json.dumps(payload['summary'], indent=2)[:1200])
    elif 'cache_summary' in payload:
        print(json.dumps(payload['cache_summary'], indent=2))
    else:
        print(json.dumps(payload, indent=2)[:1200])


=== outputs/colab_main_smoke.json ===
{
  "count": 1,
  "avg_retention_ratio": 0.5833333333333334,
  "avg_compression_ratio": 0.41666666666666663,
  "avg_compression_multiplier": 1.7142857142857142,
  "avg_budget_gap": -1.0,
  "avg_latency_ms": 0.7378280000125415,
  "p50_latency_ms": 0.7378280000125415,
  "p90_latency_ms": 0.7378280000125415
}

=== outputs/colab_baselines_smoke.json ===
{
  "config": {
    "trace_path": "data/sample_trace.jsonl",
    "methods": [
      "streamingllm",
      "chunkkv",
      "snapkv",
      "h2o"
    ],
    "budget": null,
    "theta": 0.3,
    "recent_window": 4,
    "heavy_hitter_ratio": 0.7
  },
  "results": {
    "streamingllm": {
      "summary": {
        "count": 1,
        "avg_retention_ratio": 0.6666666666666666,
        "avg_compression_ratio": 0.33333333333333337,
        "avg_compression_multiplier": 1.5,
        "avg_budget_gap": 0.0,
        "avg_latency_ms": 0.6963760000076036,
        "p50_latency_ms": 0.6963760000076036,
        "p90_

## 8. Optional Tiny HuggingFace Smoke Test

This checks model loading, attention extraction, chunk scoring, and multi-method HF-grid output. It uses `sshleifer/tiny-gpt2` and `max_new_tokens=0`, so it is a pipeline smoke test, not a quality result.

In [9]:
# 8. Optional tiny HF smoke test
tiny_records = [
    {'id': 'toy_0', 'prompt': 'Question: What is 2 plus 2?\nAnswer:', 'answer': '4'},
]
tiny_path = Path('data/colab_tiny_qa.jsonl')
with tiny_path.open('w', encoding='utf-8') as f:
    for rec in tiny_records:
        f.write(json.dumps(rec) + '\n')

dataset_spec = 'name=tiny,source=data/colab_tiny_qa.jsonl,prompt_field=prompt,answer_field=answer,id_field=id'
subprocess.run([
    sys.executable, 'scripts/run_hf_grid.py',
    '--models', 'sshleifer/tiny-gpt2',
    '--datasets', dataset_spec,
    '--budget-ratios', '0.5',
    '--thetas', '0.3',
    '--recent-windows', '4',
    '--alphas', '0.6',
    '--methods', 'fullkv,streamingllm,h2o,snapkv,chunkkv,tdc_kv',
    '--max-samples', '1',
    '--max-length', '128',
    '--max-new-tokens', '0',
    '--device', 'auto',
    '--dtype', 'auto',
    '--output', 'outputs/colab_tiny_hf_grid.json',
], check=True)

with open('outputs/colab_tiny_hf_grid.json', 'r', encoding='utf-8') as f:
    hf_payload = json.load(f)
print(json.dumps(hf_payload['summary'], indent=2)[:2000])

{
  "total_runs": 6,
  "successful_runs": 6,
  "failed_runs": 0,
  "cache_summary": {
    "count": 5,
    "avg_retention_ratio": 0.7272727272727273,
    "avg_compression_ratio": 0.27272727272727276,
    "avg_compression_multiplier": 1.5,
    "avg_budget_gap": 2.0,
    "avg_latency_ms": 0.47102580001592287,
    "p50_latency_ms": 0.43788099998209873,
    "p90_latency_ms": 0.46372300005259603
  },
  "baseline_qa_summary": {
    "count": 0,
    "exact_match": 0.0,
    "f1": 0.0
  },
  "evicted_qa_summary": {
    "count": 1,
    "exact_match": 0.0,
    "f1": 0.0
  },
  "method_summaries": {
    "fullkv": {
      "cache_summary": {
        "count": 1,
        "avg_retention_ratio": 1.0,
        "avg_compression_ratio": 0.0,
        "avg_compression_multiplier": 1.0,
        "avg_budget_gap": 0.0,
        "avg_latency_ms": 0.0,
        "p50_latency_ms": 0.0,
        "p90_latency_ms": 0.0
      },
      "qa_summary": {
        "count": 0,
        "exact_match": 0.0,
        "f1": 0.0
      }
 

## 9. Next Small Real Test

After the tiny smoke test works, try a small real model on 2 to 5 examples. Use this only on GPU.

Suggested first model:

- `Qwen/Qwen2.5-0.5B-Instruct`

Then, if it works:

- `Qwen/Qwen2.5-1.5B-Instruct`
- `Qwen/Qwen2.5-3B-Instruct`

Do not start with 7B/8B until the full pipeline passes on small models.

In [13]:
import subprocess, sys

cmd = [
    sys.executable, 'scripts/run_hf_grid.py',
    '--models', 'Qwen/Qwen2.5-0.5B-Instruct',
    '--datasets', 'name=gsm8k,source=openai/gsm8k,config=main,split=test,prompt_field=question,answer_field=answer',
    '--budget-ratios', '0.5',
    '--thetas', '0.3',
    '--recent-windows', '16',
    '--alphas', '0.6',
    '--methods', 'tdc_kv',
    '--max-samples', '1',
    '--max-length', '512',
    '--max-new-tokens', '0',
    '--device', 'auto',
    '--dtype', 'auto',
    '--output', 'outputs/qwen05b_prefill_only.json',
]

result = subprocess.run(cmd, text=True, capture_output=True)
print("RETURN CODE:", result.returncode)
print("\nSTDOUT:\n", result.stdout[-4000:])
print("\nSTDERR:\n", result.stderr[-8000:])

RETURN CODE: 0

STDOUT:
 Running grid search for 1 models and 1 datasets...
Results saved to outputs/qwen05b_prefill_only.json


STDERR:

Generating train split: 100%|██████████| 7473/7473 [00:00<00:00, 158210.51 examples/s]

Generating test split: 100%|██████████| 1319/1319 [00:00<00:00, 234518.31 examples/s]

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 378.27it/s]
/content/Tier-based-KV-cache-with-Dependency-Aware-Chunk-Scoring/src/core/evictor.py:147: UserWarning: Unable to satisfy budget 32 without evicting Tier-2 chunks. Evicted 20 tokens instead of the required 33. The resulting cache will exceed the requested budget to preserve hard-protected tokens.
  warnings.warn(



In [ ]:
cmd = [
    sys.executable, 'scripts/run_hf_grid.py',
    '--models', 'Qwen/Qwen2.5-0.5B-Instruct',
    '--datasets', 'name=gsm8k,source=openai/gsm8k,config=main,split=test,prompt_field=question,answer_field=answer',
    '--budget-ratios', '0.5,0.25',
    '--thetas', '0.3',
    '--recent-windows', '16',
    '--alphas', '0.6',
    '--methods', 'fullkv,tdc_kv',
    '--max-samples', '5',
    '--max-length', '1024',
    '--max-new-tokens', '32',
    '--device', 'auto',
    '--dtype', 'auto',
    '--output', 'outputs/qwen05b_gsm8k_tdc_smoke.json',
]

result = subprocess.run(cmd, text=True, capture_output=True)
print("RETURN CODE:", result.returncode)
print("\nSTDOUT:\n", result.stdout[-4000:])
print("\nSTDERR:\n", result.stderr[-8000:])